In [1]:
import os
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
from torchvision import transforms, models
from transformers import CLIPProcessor, CLIPVisionModel, ViTModel, ViTImageProcessor
import re
from collections import Counter
from tqdm import tqdm
import argparse
import random

# Tối ưu hóa môi trường
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# =========================================================================
# 1. ENCODERS
# =========================================================================
class CNNEncoder(nn.Module):
    def __init__(self, model_name="resnet50"):
        super().__init__()
        if model_name == "resnet50":
            # ResNet50
            resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
            modules = list(resnet.children())[:-2]
            self.model = nn.Sequential(*modules)
            self.out_dim = 2048
        elif model_name == "efficientnet_b3":
            # EfficientNet-B3
            effnet = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
            self.model = effnet.features
            self.out_dim = 1536
        else:
            raise ValueError(f"Model {model_name} not supported by CNNEncoder")
            
        for param in self.model.parameters():
            param.requires_grad = False # Freeze encoder

    def forward(self, images):
        features = self.model(images) # [B, C, H, W]
        features = features.permute(0, 2, 3, 1) # [B, H, W, C]
        features = features.view(features.size(0), -1, features.size(-1)) # [B, H*W, C]
        return features

class ViTEncoderModel(nn.Module):
    def __init__(self, model_name="google/vit-base-patch16-224-in21k"):
        super().__init__()
        self.model = ViTModel.from_pretrained(model_name)
        self.out_dim = 768
        for param in self.model.parameters():
            param.requires_grad = False
            
    def forward(self, images):
        outputs = self.model(pixel_values=images)
        return outputs.last_hidden_state

class CLIPEncoderModel(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch16"):
        super().__init__()
        self.model = CLIPVisionModel.from_pretrained(model_name)
        self.out_dim = 768
        for param in self.model.parameters():
            param.requires_grad = False
            
    def forward(self, images):
        outputs = self.model(pixel_values=images)
        return outputs.last_hidden_state

class CustomTokenizer:
    PAD_TOKEN = "<pad>"
    BOS_TOKEN = "<bos>"
    EOS_TOKEN = "<eos>"
    UNK_TOKEN = "<unk>"

    def __init__(self, captions=None, min_freq=2):
        self.min_freq = min_freq

        self.word_to_id = {
            self.PAD_TOKEN: 0,
            self.BOS_TOKEN: 1,
            self.EOS_TOKEN: 2,
            self.UNK_TOKEN: 3
        }

        self.id_to_word = {
            0: self.PAD_TOKEN,
            1: self.BOS_TOKEN,
            2: self.EOS_TOKEN,
            3: self.UNK_TOKEN
        }

        if captions is not None:
            self.build_vocab(captions)

    def tokenize(self, text):
        text = str(text).lower().strip()

        # Lấy từ và số, bỏ dấu câu
        return re.findall(r"\b\w+\b", text)

    def build_vocab(self, captions):
        counter = Counter()

        for caption in captions:
            tokens = self.tokenize(caption)
            counter.update(tokens)

        next_id = len(self.word_to_id)

        for word, frequency in counter.items():
            if frequency >= self.min_freq:
                self.word_to_id[word] = next_id
                self.id_to_word[next_id] = word
                next_id += 1

        print(f"[*] Custom vocabulary size: {len(self.word_to_id)}")

    def encode(self, text, max_length):
        tokens = self.tokenize(text)

        token_ids = [self.bos_token_id]

        for token in tokens:
            token_id = self.word_to_id.get(token, self.unk_token_id)
            token_ids.append(token_id)

        token_ids.append(self.eos_token_id)

        # Cắt bớt nếu caption quá dài
        if len(token_ids) > max_length:
            token_ids = token_ids[:max_length]
            token_ids[-1] = self.eos_token_id

        attention_mask = [1] * len(token_ids)

        # Padding
        padding_length = max_length - len(token_ids)

        token_ids.extend([self.pad_token_id] * padding_length)
        attention_mask.extend([0] * padding_length)

        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(attention_mask, dtype=torch.long)
        )

    def decode(self, token_ids):
        words = []

        for token_id in token_ids:
            token_id = int(token_id)

            if token_id == self.eos_token_id:
                break

            if token_id in [self.pad_token_id, self.bos_token_id]:
                continue

            words.append(
                self.id_to_word.get(token_id, self.UNK_TOKEN)
            )

        return " ".join(words)

    @property
    def vocab_size(self):
        return len(self.word_to_id)

    @property
    def pad_token_id(self):
        return self.word_to_id[self.PAD_TOKEN]

    @property
    def bos_token_id(self):
        return self.word_to_id[self.BOS_TOKEN]

    @property
    def eos_token_id(self):
        return self.word_to_id[self.EOS_TOKEN]

    @property
    def unk_token_id(self):
        return self.word_to_id[self.UNK_TOKEN]

# =========================================================================
# 2. DATASET
# =========================================================================
class Flickr8kDataset(Dataset):
    def __init__(
        self,
        csv_path,
        img_dir,
        tokenizer,
        encoder_type="resnet50",
        max_length=32,
        is_train=True
    ):
        self.img_dir = img_dir
        self.max_length = max_length
        self.is_train = is_train
        self.encoder_type = encoder_type

        # Text Tokenizer
        self.tokenizer = tokenizer

        # Image Processor
        if encoder_type in ["resnet50", "efficientnet_b3"]:
            # Standard Torchvision Transform
            normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            if self.is_train:
                self.transform = transforms.Compose([
                    transforms.Resize((224, 224)),
                    transforms.RandomHorizontalFlip(p=0.5),
                    transforms.ColorJitter(brightness=0.2, contrast=0.2),
                    transforms.ToTensor(),
                    normalize
                ])
            else:
                self.transform = transforms.Compose([
                    transforms.Resize((224, 224)),
                    transforms.ToTensor(),
                    normalize
                ])
            self.processor = None
        elif encoder_type == "vit_b_16":
            self.processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
            self.transform = None
        elif encoder_type == "clip_vit_b_16":
            self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
            self.transform = None
            
        # Optional augmentations for processor-based methods
        self.color_jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2) if self.is_train else None

        # Load Flickr8k captions.txt format: image,caption
        df = pd.read_csv(csv_path, sep=',')
        if 'image' not in df.columns or 'caption' not in df.columns:
            # Maybe it is image_name, comment
            df.columns = ['image', 'caption']
            
        self.image_files = df['image'].tolist()
        self.captions = df['caption'].tolist()

        print(f"[*] Dataset Flickr8k loaded: {len(self.image_files)} samples.")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        
        if self.encoder_type in ["resnet50", "efficientnet_b3"]:
            pixel_values = self.transform(image)
        else:
            if self.color_jitter:
                image = self.color_jitter(image)
            pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze(0)

        # Text padding
        caption_text = str(self.captions[idx])

        input_ids, attention_mask = self.tokenizer.encode(
            caption_text,
            max_length=self.max_length
        )
        
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

def collate_fn_custom(batch):
    return {
        "pixel_values": torch.stack([item["pixel_values"] for item in batch]),
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "attention_mask": torch.stack([item["attention_mask"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }

# =========================================================================
# 3. MODEL ARCHITECTURE
# =========================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)


class UniversalTransformerCaptioner(nn.Module):
    def __init__(self,
                 encoder_type="resnet50",
                 vocab_size=50257,
                 d_model=512,
                 nhead=8,
                 num_layers=6,
                 dim_feedforward=2048,
                 dropout=0.1):
        super().__init__()

        # 1. Encoder
        if encoder_type == "resnet50":
            self.encoder = CNNEncoder("resnet50")
        elif encoder_type == "efficientnet_b3":
            self.encoder = CNNEncoder("efficientnet_b3")
        elif encoder_type == "vit_b_16":
            self.encoder = ViTEncoderModel()
        elif encoder_type == "clip_vit_b_16":
            self.encoder = CLIPEncoderModel()
        else:
            raise ValueError(f"Unknown encoder type: {encoder_type}")

        encoder_hidden_size = self.encoder.out_dim

        # Project output to transformer d_model
        self.enc_projection = nn.Linear(encoder_hidden_size, d_model)

        # 2. Transformer Decoder
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward, dropout=dropout,
                                                   batch_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        # 3. Output projection
        self.fc_out = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )
        
        self.fc_out.weight = self.embedding.weight
        self.loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, pixel_values, input_ids, labels=None, attention_mask=None):
        # 1. Encode Image
        with torch.no_grad():
            image_embeds = self.encoder(pixel_values)

        image_embeds = self.enc_projection(image_embeds)

        # 2. Decode Text
        seq_len = input_ids.size(1)
        tgt_mask = self.generate_square_subsequent_mask(seq_len).to(input_ids.device)

        # Embedding + Positional Encoding
        tgt_emb = self.embedding(input_ids)
        tgt_emb = tgt_emb.transpose(0, 1)
        tgt_emb = self.pos_encoder(tgt_emb)
        tgt_emb = tgt_emb.transpose(0, 1)

        tgt_key_padding_mask = None
        if attention_mask is not None:
            tgt_key_padding_mask = (attention_mask == 0)

        output = self.decoder(
            tgt=tgt_emb,
            memory=image_embeds,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )

        logits = self.fc_out(output)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = self.loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        return loss, logits

# =========================================================================
# 4. TRAINING LOOP
# =========================================================================
def _unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def save_model(model, optimizer=None, scaler=None, epoch=0, best_val_loss=None,
               save_path="captioner_checkpoint.pth"):
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": _unwrap_model(model).state_dict(),
        "best_val_loss": best_val_loss,
    }
    if optimizer is not None:
        checkpoint["optimizer_state_dict"] = optimizer.state_dict()
    if scaler is not None:
        checkpoint["scaler_state_dict"] = scaler.state_dict()
    torch.save(checkpoint, save_path)
    print(f"[*] Đã lưu checkpoint: {save_path}")

def load_model(model, checkpoint_path, optimizer=None, scaler=None, device='cuda', strict=True):
    if not checkpoint_path or not os.path.exists(checkpoint_path):
        return 0, float('inf')
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if "model_state_dict" in checkpoint:
        _unwrap_model(model).load_state_dict(checkpoint["model_state_dict"], strict=strict)
    else:
        _unwrap_model(model).load_state_dict(checkpoint, strict=strict)
        return 0, float('inf')
    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scaler is not None and "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch = checkpoint.get("epoch", 0)
    best_val_loss = checkpoint.get("best_val_loss", float('inf'))
    print(f"[*] Đã load checkpoint: {checkpoint_path} | resume từ epoch {start_epoch + 1}")
    return start_epoch, best_val_loss

def train_model(model, train_loader, val_loader=None, epochs=10, learning_rate=1e-4,
                device='cuda', checkpoint_dir="checkpoints", resume_from=None):
    model.to(device)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
    scaler = torch.cuda.amp.GradScaler()

    best_val_loss = float('inf')
    start_epoch = 0

    if resume_from:
        start_epoch, best_val_loss = load_model(
            model=model, checkpoint_path=resume_from,
            optimizer=optimizer, scaler=scaler, device=device
        )

    for epoch in range(start_epoch, epochs):
        model.train()
        total_train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Train")
        for batch in pbar:
            pixel_values = batch["pixel_values"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            with torch.autocast(device_type=device, dtype=torch.float16):
                loss, _ = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids,
                    labels=labels,
                    attention_mask=attention_mask
                )
            if loss.dim() > 0: loss = loss.mean()

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_train_loss += loss.item()
            pbar.set_postfix({"Train Loss": f"{loss.item():.4f}"})

        avg_train_loss = total_train_loss / len(train_loader)

        if val_loader is not None:
            model.eval()
            total_val_loss = 0
            with torch.no_grad():
                val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} - Val")
                for batch in val_pbar:
                    pixel_values = batch["pixel_values"].to(device)
                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    labels = batch["labels"].to(device)

                    with torch.autocast(device_type=device, dtype=torch.float16):
                        loss, _ = model(
                            pixel_values=pixel_values, input_ids=input_ids,
                            labels=labels, attention_mask=attention_mask
                        )
                    if loss.dim() > 0: loss = loss.mean()
                    total_val_loss += loss.item()
                    val_pbar.set_postfix({"Val Loss": f"{loss.item():.4f}"})

            avg_val_loss = total_val_loss / len(val_loader)
            print(f"Epoch {epoch+1}/{epochs} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                save_model(model, optimizer, scaler, epoch + 1, best_val_loss,
                           os.path.join(checkpoint_dir, "best_captioner.pth"))
        else:
            print(f"Epoch {epoch+1}/{epochs} | Train: {avg_train_loss:.4f}")

        save_model(model, optimizer, scaler, epoch + 1, best_val_loss,
                   os.path.join(checkpoint_dir, "last_captioner.pth"))

def split_caption_dataset(captions_csv, output_dir="splits", seed=42,
                          train_ratio=0.75, val_ratio=0.125, test_ratio=0.125):
    """Split Flickr8k captions by unique image to avoid data leakage."""
    ratio_sum = train_ratio + val_ratio + test_ratio
    if abs(ratio_sum - 1.0) > 1e-9:
        raise ValueError(f"Ratios must sum to 1.0, got {ratio_sum}")

    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(captions_csv)
    if "image" not in df.columns or "caption" not in df.columns:
        df = df.iloc[:, :2].copy()
        df.columns = ["image", "caption"]

    random.seed(seed)
    images = df["image"].unique().tolist()
    random.shuffle(images)
    nimages = df["image"].nunique()
    train_split = round(train_ratio * nimages)
    val_split = round(val_ratio * nimages)
    test_split = nimages - train_split - val_split

    train_images = images[:train_split]
    val_images = images[train_split:train_split + val_split]
    test_images = images[train_split + val_split:]

    split_paths = {
        "train": os.path.join(output_dir, "train_captions.csv"),
        "val": os.path.join(output_dir, "val_captions.csv"),
        "test": os.path.join(output_dir, "test_captions.csv"),
    }

    split_defs = {
        "train": train_images,
        "val": val_images,
        "test": test_images,
    }

    for split_name, split_images in split_defs.items():
        split_df = df[df["image"].isin(split_images)].copy()
        split_df.to_csv(split_paths[split_name], index=False)
        print(f"{split_name}: {len(split_images)} images, {len(split_df)} captions")

    print(f"test_split expected: {test_split} images")

    return split_paths

if __name__ == '__main__':
    # -------------------------------------------------------------------------
    # PARAMETERS / CONFIG
    # -------------------------------------------------------------------------
    # Lựa chọn 1 trong 4 models: 
    # 'resnet50', 'efficientnet_b3', 'vit_b_16', 'clip_vit_b_16'
    ENCODER_TYPE = "efficientnet_b3"
    
    SOURCE_CSV = "/kaggle/input/datasets/adityajn105/flickr8k/captions.txt"
    SPLIT_DIR = "splits"
    TRAIN_CSV = os.path.join(SPLIT_DIR, "train_captions.csv")
    TRAIN_IMG_DIR = "/kaggle/input/datasets/adityajn105/flickr8k/Images"
    VAL_CSV = os.path.join(SPLIT_DIR, "val_captions.csv")
    VAL_IMG_DIR = TRAIN_IMG_DIR
    
    CHECKPOINT_DIR = "checkpoints"
    BATCH_SIZE = 32
    EPOCHS = 20
    LEARNING_RATE = 1e-4

    # -------------------------------------------------------------------------
    if os.path.exists(SOURCE_CSV) and not os.path.exists(TRAIN_CSV):
        print("Chia dataset Flickr8k theo tỉ lệ train/val/test = 8/1/1...")
        split_caption_dataset(SOURCE_CSV, output_dir=SPLIT_DIR, seed=42)

    if not os.path.exists(TRAIN_CSV) or not os.path.exists(TRAIN_IMG_DIR):
        print(f"Vui lòng cập nhật SOURCE_CSV và TRAIN_IMG_DIR cho Flickr8k!")
    else:
        print(f"Khởi tạo model với encoder: {ENCODER_TYPE}")
        caption_df = pd.read_csv(TRAIN_CSV)

        if "image" not in caption_df.columns or "caption" not in caption_df.columns:
            caption_df.columns = ["image", "caption"]
        
        train_captions = caption_df["caption"].astype(str).tolist()
        
        tokenizer = CustomTokenizer(
            captions=train_captions,
            min_freq=2
        )
        
        vocab_size = tokenizer.vocab_size
        
        print(f"Vocabulary size: {vocab_size}")

        train_dataset = Flickr8kDataset(
            csv_path=TRAIN_CSV,
            img_dir=TRAIN_IMG_DIR,
            tokenizer=tokenizer,
            encoder_type=ENCODER_TYPE,
            max_length=32,
            is_train=True
        )

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_SIZE, shuffle=True,
            collate_fn=collate_fn_custom, pin_memory=True, num_workers=0
        )

        val_loader = None
        if VAL_CSV and VAL_IMG_DIR and os.path.exists(VAL_CSV):
            val_dataset = Flickr8kDataset(
                csv_path=VAL_CSV,
                img_dir=VAL_IMG_DIR,
                tokenizer=tokenizer,
                encoder_type=ENCODER_TYPE,
                max_length=32,
                is_train=False
            )
            val_loader = DataLoader(
                val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                collate_fn=collate_fn_custom, pin_memory=True, num_workers=0
            )

        model = UniversalTransformerCaptioner(
            encoder_type=ENCODER_TYPE,
            vocab_size=vocab_size,
            d_model=512,
            nhead=8,
            num_layers=6
        )

        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model.to(device)
    

        print(f"Bắt đầu huấn luyện trên {device}...")
        train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=EPOCHS,
            learning_rate=LEARNING_RATE,
            device=device,
            checkpoint_dir=os.path.join(CHECKPOINT_DIR, ENCODER_TYPE)
        )


Khởi tạo model với encoder: efficientnet_b3
[*] Custom vocabulary size: 5156
Vocabulary size: 5156
[*] Dataset Flickr8k loaded: 40455 samples.
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 230MB/s]
/tmp/ipykernel_24/1392365134.py:444: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Bắt đầu huấn luyện trên cuda...


Epoch 1/20 - Train:   0%|          | 0/1265 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
Epoch 1/20 - Train: 100%|██████████| 1265/1265 [07:11<00:00,  2.93it/s, Train Loss=8.4579]


Epoch 1/20 | Train: 12.3259
[*] Đã lưu checkpoint: checkpoints/efficientnet_b3/last_captioner.pth


...

Epoch 18/20 | Train: 2.2203
[*] Đã lưu checkpoint: checkpoints/efficientnet_b3/last_captioner.pth


Epoch 19/20 - Train: 100%|██████████| 1265/1265 [06:28<00:00,  3.25it/s, Train Loss=2.0259]


Epoch 19/20 | Train: 2.1512
[*] Đã lưu checkpoint: checkpoints/efficientnet_b3/last_captioner.pth


Epoch 20/20 - Train: 100%|██████████| 1265/1265 [06:28<00:00,  3.26it/s, Train Loss=2.4299]


Epoch 20/20 | Train: 2.0897
[*] Đã lưu checkpoint: checkpoints/efficientnet_b3/last_captioner.pth


In [ ]:
# =========================================================================
# 5. EVALUATION + TEST CAPTION GENERATION
# =========================================================================
# Optional packages for full metrics:
# !pip install nltk pycocoevalcap

from pathlib import Path

try:
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    from nltk.translate.meteor_score import meteor_score
except ImportError:
    corpus_bleu = SmoothingFunction = meteor_score = None

def preprocess_single_image(image_path, encoder_type):
    image = Image.open(image_path).convert("RGB")
    if encoder_type in ["resnet50", "efficientnet_b3"]:
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        return transform(image)
    if encoder_type == "vit_b_16":
        processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
        return processor(images=image, return_tensors="pt").pixel_values.squeeze(0)
    if encoder_type == "clip_vit_b_16":
        processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
        return processor(images=image, return_tensors="pt").pixel_values.squeeze(0)
    raise ValueError(f"Unknown encoder_type: {encoder_type}")

def generate_caption(model, image_tensor, tokenizer, device, max_length=32):
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)
    generated = torch.tensor([[tokenizer.bos_token_id]], dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_length - 1):
            attention_mask = torch.ones_like(generated, device=device)
            _, logits = model(
                pixel_values=image_tensor,
                input_ids=generated,
                attention_mask=attention_mask
            )
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(generated.squeeze(0).detach().cpu().tolist())

def generate_captions_from_folder(model, test_img_dir, tokenizer, encoder_type, device,
                                  test_csv=None, max_length=32, output_csv="test_predictions.csv"):
    test_img_dir = Path(test_img_dir)
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    references_by_image = {}
    if test_csv is not None and os.path.exists(test_csv):
        ref_df = pd.read_csv(test_csv)
        if "image" not in ref_df.columns or "caption" not in ref_df.columns:
            ref_df.columns = ["image", "caption"]
        for image_name, group in ref_df.groupby("image"):
            references_by_image[str(image_name)] = group["caption"].astype(str).tolist()
        image_names = list(references_by_image.keys())
    else:
        image_names = sorted([p.name for p in test_img_dir.iterdir() if p.suffix.lower() in image_exts])

    rows = []
    for image_name in tqdm(image_names, desc="Generating captions"):
        image_path = test_img_dir / image_name
        if not image_path.exists():
            print(f"[!] Missing image: {image_path}")
            continue
        image_tensor = preprocess_single_image(image_path, encoder_type)
        pred_caption = generate_caption(model, image_tensor, tokenizer, device, max_length=max_length)
        rows.append({
            "image": image_name,
            "generated_caption": pred_caption,
            "reference_captions": " | ".join(references_by_image.get(image_name, []))
        })

    pred_df = pd.DataFrame(rows)
    pred_df.to_csv(output_csv, index=False)
    print(f"[*] Saved generated captions to: {output_csv}")
    return pred_df, references_by_image

def _metric_tokens(text):
    return re.findall(r"\b\w+\b", str(text).lower())

def _rouge_l_f1(reference_tokens, hypothesis_tokens):
    if not reference_tokens or not hypothesis_tokens:
        return 0.0
    dp = [[0] * (len(hypothesis_tokens) + 1) for _ in range(len(reference_tokens) + 1)]
    for i, ref_token in enumerate(reference_tokens, 1):
        for j, hyp_token in enumerate(hypothesis_tokens, 1):
            if ref_token == hyp_token:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    lcs = dp[-1][-1]
    precision = lcs / len(hypothesis_tokens)
    recall = lcs / len(reference_tokens)
    return 0.0 if precision + recall == 0 else (2 * precision * recall) / (precision + recall)

def evaluate_caption_metrics(pred_df, references_by_image):
    if corpus_bleu is None or meteor_score is None:
        raise ImportError("Please install nltk first: pip install nltk")

    eval_rows = []
    for row in pred_df.itertuples(index=False):
        refs = references_by_image.get(row.image, [])
        if refs:
            eval_rows.append((row.image, refs, row.generated_caption))

    if not eval_rows:
        print("[!] No reference captions found. Metrics skipped; generated captions are still saved.")
        return {}

    refs_tokenized = [[_metric_tokens(ref) for ref in refs] for _, refs, _ in eval_rows]
    hyps_tokenized = [_metric_tokens(pred) for _, _, pred in eval_rows]
    smooth = SmoothingFunction().method1

    bleu_1 = corpus_bleu(refs_tokenized, hyps_tokenized, weights=(1.0, 0, 0, 0), smoothing_function=smooth)
    bleu_4 = corpus_bleu(refs_tokenized, hyps_tokenized, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
    meteor = sum(meteor_score(refs, hyp, wordnet=None) for refs, hyp in zip(refs_tokenized, hyps_tokenized)) / len(eval_rows)
    rouge_l = sum(max(_rouge_l_f1(_metric_tokens(ref), hyp) for ref in refs)
                  for (_, refs, _), hyp in zip(eval_rows, hyps_tokenized)) / len(eval_rows)

    cider = None
    try:
        from pycocoevalcap.cider.cider import Cider
        gts = {image: refs for image, refs, _ in eval_rows}
        res = {image: [pred] for image, _, pred in eval_rows}
        cider, _ = Cider().compute_score(gts, res)
    except Exception as exc:
        print(f"[!] CIDEr skipped. Install pycocoevalcap to enable it. Reason: {exc}")

    metrics = {
        "BLEU-1": bleu_1,
        "BLEU-4": bleu_4,
        "METEOR": meteor,
        "ROUGE-L": rouge_l,
        "CIDEr": cider
    }
    print(pd.DataFrame([metrics]).to_string(index=False))
    return metrics

def load_selected_captioner(checkpoint_path, encoder_type, tokenizer, device=None,
                            d_model=512, nhead=8, num_layers=6):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = UniversalTransformerCaptioner(
        encoder_type=encoder_type,
        vocab_size=tokenizer.vocab_size,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers
    )
    model.to(device)
    load_model(model, checkpoint_path, device=device)
    model.eval()
    return model, device

# -----------------------------
# RUN THIS AFTER TRAINING / TOKENIZER CREATION
# -----------------------------
# 1) Choose the same encoder used for training.
# 2) Set TEST_IMG_DIR to your test image folder.
# 3) Set TEST_CSV if you have test captions; leave None for generation only.

SELECTED_ENCODER_TYPE = "efficientnet_b3"
SELECTED_CHECKPOINT = os.path.join("checkpoints", SELECTED_ENCODER_TYPE, "best_captioner.pth")
TEST_IMG_DIR = "/kaggle/input/datasets/adityajn105/flickr8k/Images"
TEST_CSV = "splits/test_captions.csv"

# If model/tokenizer already exist in memory after training, you can skip load_selected_captioner.
# eval_model, eval_device = load_selected_captioner(SELECTED_CHECKPOINT, SELECTED_ENCODER_TYPE, tokenizer)
# pred_df, refs = generate_captions_from_folder(eval_model, TEST_IMG_DIR, tokenizer, SELECTED_ENCODER_TYPE, eval_device, TEST_CSV)
# metrics = evaluate_caption_metrics(pred_df, refs)
